### Oversampled DLCNN with Image Rotations and Flipping

In [10]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from collections import defaultdict
import random
import os

seed = 3888
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

img_width, img_height = 100, 100
batch_size = 512
AUTOTUNE = tf.data.AUTOTUNE

tumour_folders = ["Invasive_Tumor", "DCIS_1", "DCIS_2", "Prolif_Invasive_Tumor"]
immune_folders = ["CD4+_T_Cells", "CD8+_T_Cells", "B_Cells", "Stromal"]
sub_label_list = tumour_folders + immune_folders
sub_label_to_index = {name: idx for idx, name in enumerate(sub_label_list)}

def get_file_label_pairs(folders, main_label):
    pairs = []
    for folder in folders:
        files = tf.io.gfile.glob(os.path.join(folder, "*.png"))
        for f in files:
            sub_label = os.path.basename(folder)
            pairs.append((f, main_label, sub_label))
    return pairs

tumour_files = get_file_label_pairs(tumour_folders, main_label=1)
immune_files = get_file_label_pairs(immune_folders, main_label=0)
all_file_label_pairs = tumour_files + immune_files
print(f"Total images before balancing: {len(all_file_label_pairs)}")

def create_balanced_oversampled_dataset(file_label_pairs):
    label_to_files = defaultdict(list)
    for f, main_label, sub_label in file_label_pairs:
        label_to_files[sub_label].append((f, main_label, sub_label))
    
    max_class_size = max(len(files) for files in label_to_files.values())
    print(f"Max class size to balance to: {max_class_size}")

    oversampled = []
    for sub_label, files in label_to_files.items():
        originals = files.copy()
        oversampled.extend((f, m, s, False) for f, m, s in originals)

        extra_needed = max_class_size - len(originals)
        augmented_files = random.choices(originals, k=extra_needed)
        oversampled.extend((f, m, s, True) for f, m, s in augmented_files)

    random.shuffle(oversampled)
    return oversampled

oversampled_pairs = create_balanced_oversampled_dataset(all_file_label_pairs)
print(f"Total images after oversampling: {len(oversampled_pairs)}")

def augment_image(img):
    angles = [0, 90, 180, 270]
    angle = random.choice(angles)
    
    if angle == 90:
        img = tf.image.rot90(img, k=1)
    elif angle == 180:
        img = tf.image.rot90(img, k=2)
    elif angle == 270:
        img = tf.image.rot90(img, k=3)

    flip_type = random.choice(['none', 'horizontal', 'vertical', 'both'])
    if flip_type == 'horizontal':
        img = tf.image.flip_left_right(img)
    elif flip_type == 'vertical':
        img = tf.image.flip_up_down(img)
    elif flip_type == 'both':
        img = tf.image.flip_left_right(img)
        img = tf.image.flip_up_down(img)

    return img

def decode_img(img_path, augment=False):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [img_width, img_height])
    img = img / 255.0
    if augment:
        img = augment_image(img)
    return img

def prepare_dataset(pairs):
    file_paths, main_labels, sub_labels, augment_flags = [], [], [], []

    for f, m, s, a in pairs:
        file_paths.append(f)
        main_labels.append(m)
        sub_labels.append(sub_label_to_index[s])
        augment_flags.append(a)

    file_paths = np.array(file_paths)
    main_labels = np.array(main_labels)
    sub_labels = np.array(sub_labels)
    augment_flags = np.array(augment_flags)

    train_files, test_files, train_main, test_main, train_sub, test_sub, train_aug, test_aug = train_test_split(
        file_paths, main_labels, sub_labels, augment_flags, test_size=0.2, stratify=sub_labels, random_state=seed
    )

    def process_path(file_path, main_label, sub_label, augment):
        img = decode_img(file_path, augment)
        return img, {"binary_output": main_label, "multiclass_output": sub_label}

    train_ds = tf.data.Dataset.from_tensor_slices((train_files, train_main, train_sub, train_aug))
    train_ds = train_ds.map(process_path, num_parallel_calls=AUTOTUNE).shuffle(1000).batch(batch_size).prefetch(AUTOTUNE)

    test_ds = tf.data.Dataset.from_tensor_slices((test_files, test_main, test_sub, test_aug))
    test_ds = test_ds.map(process_path, num_parallel_calls=AUTOTUNE).batch(batch_size).prefetch(AUTOTUNE)

    return train_ds, test_ds

def build_multitask_model():
    inputs = tf.keras.Input(shape=(img_width, img_height, 3))
    x = tf.keras.layers.Conv2D(32, (3,3), activation='relu')(inputs)
    x = tf.keras.layers.MaxPooling2D((2,2))(x)
    x = tf.keras.layers.Conv2D(64, (3,3), activation='relu')(x)
    x = tf.keras.layers.MaxPooling2D((2,2))(x)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)

    binary_output = tf.keras.layers.Dense(1, activation='sigmoid', name="binary_output")(x)
    multiclass_logits = tf.keras.layers.Dense(8, name="raw_multiclass_logits")(x)
    scaled_logits = tf.keras.layers.Multiply()([multiclass_logits, binary_output])
    multiclass_output = tf.keras.layers.Activation('softmax', name="multiclass_output")(scaled_logits)

    model = tf.keras.Model(inputs=inputs, outputs=[binary_output, multiclass_output])

    model.compile(
        optimizer='adam',
        loss={
            "binary_output": "binary_crossentropy",
            "multiclass_output": "sparse_categorical_crossentropy"
        },
        metrics={
            "binary_output": "accuracy",
            "multiclass_output": "accuracy"
        }
    )

    return model

def train_and_evaluate(train_ds, test_ds, title="Model"):
    model = build_multitask_model()
    model.summary()

    history = model.fit(train_ds, epochs=10, validation_data=test_ds)

    results = model.evaluate(test_ds)
    print(f"\nTest results - {title}:")
    print(f"Binary Accuracy: {results[3]:.4f}")
    print(f"Multiclass Accuracy: {results[4]:.4f}")

    y_true, y_pred = [], []

    for images, labels in test_ds:
        preds = model.predict(images, verbose=0)
        bin_preds = preds[0].flatten()
        multi_preds = preds[1]

        true_bin = labels['binary_output'].numpy().flatten()
        true_multi = labels['multiclass_output'].numpy().flatten()

        tumour_indices = np.where(true_bin == 1)[0]
        if len(tumour_indices) > 0:
            y_true.extend(true_multi[tumour_indices])
            y_pred.extend(np.argmax(multi_preds[tumour_indices], axis=1))

    if y_true and y_pred:
        y_true_tumour = np.array(y_true)
        y_pred_tumour = np.array(y_pred)

    tumour_labels = [0, 1, 2, 3]
    
    # mask = np.isin(y_true, tumour_labels)
    # y_true_tumour = y_true[mask]
    # y_pred_tumour = y_pred[mask]
    
    print("\nClassification Report (Tumour cells only):")
    print(classification_report(
        y_true_tumour, 
        y_pred_tumour, 
        labels=tumour_labels,
        target_names=tumour_folders
    ))

    # valid_indices = y_true_tumour < 4
    # y_true_tumour = y_true_tumour[valid_indices]
    # y_pred_tumour = y_pred_tumour[valid_indices]

    cm = confusion_matrix(y_true_tumour, y_pred_tumour)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=tumour_folders, yticklabels=tumour_folders)
    plt.title(f'Confusion Matrix - {title}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

    return model

train_ds_over, test_ds_over = prepare_dataset(oversampled_pairs)

print("\n--- Training with OVERSAMPLING ---")
model_over = train_and_evaluate(train_ds_over, test_ds_over, title="Oversampling")

Total images before balancing: 124622
Max class size to balance to: 41424
Total images after oversampling: 331392

--- Training with OVERSAMPLING ---


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)    │ (None, 100, 100, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_10 (Conv2D)            │ (None, 98, 98, 32)        │             896 │ input_layer_5[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d_10              │ (None, 49, 49, 32)        │               0 │ conv2d_10[0][0]            │
│ (MaxPooling2D)                │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_11 (Conv2D)            │ (None, 47, 47, 64)        │          18,496 │ max_pooling2d_10[0][0]     │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d_11              │ (None, 23, 23, 64)        │               0 │ conv2d_11[0][0]            │
│ (MaxPooling2D)                │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ flatten_5 (Flatten)           │ (None, 33856)             │               0 │ max_pooling2d_11[0][0]     │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_5 (Dense)               │ (None, 128)               │       4,333,696 │ flatten_5[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ binary_output (Dense)         │ (None, 1)                 │             129 │ dense_5[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ raw_multiclass_logits (Dense) │ (None, 8)                 │           1,032 │ dense_5[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ multiply_5 (Multiply)         │ (None, 8)                 │               0 │ raw_multiclass_logits[0][… │
│                               │                           │                 │ binary_output[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ multiclass_output             │ (None, 8)                 │               0 │ multiply_5[0][0]           │
│ (Activation)                  │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 4,354,249 (16.61 MB)

 Trainable params: 4,354,249 (16.61 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
518/518 ━━━━━━━━━━━━━━━━━━━━ 447s 856ms/step - binary_output_accuracy: 0.7630 - binary_output_loss: 0.5156 - loss: 2.4084 - multiclass_output_accuracy: 0.2609 - multiclass_output_loss: 1.8928 - val_binary_output_accuracy: 0.8634 - val_binary_output_loss: 0.3635 - val_loss: 1.9311 - val_multiclass_output_accuracy: 0.3770 - val_multiclass_output_loss: 1.5678
Epoch 2/10
518/518 ━━━━━━━━━━━━━━━━━━━━ 397s 765ms/step - binary_output_accuracy: 0.8835 - binary_output_loss: 0.3233 - loss: 1.8151 - multiclass_output_accuracy: 0.3928 - multiclass_output_loss: 1.4918 - val_binary_output_accuracy: 0.8776 - val_binary_output_loss: 0.3277 - val_loss: 1.8199 - val_multiclass_output_accuracy: 0.3969 - val_multiclass_output_loss: 1.4925
Epoch 3/10
518/518 ━━━━━━━━━━━━━━━━━━━━ 363s 699ms/step - binary_output_accuracy: 0.9002 - binary_output_loss: 0.2893 - loss: 1.6813 - multiclass_output_accuracy: 0.4359 - multiclass_output_loss: 1.3920 - val_binary_output_accuracy: 0.8799 - val_binary_output_

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (130,) + inhomogeneous part.

### CLAHE

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from collections import defaultdict
import random
import os
import cv2

seed = 3888
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

img_width, img_height = 100, 100
batch_size = 512
AUTOTUNE = tf.data.AUTOTUNE

tumour_folders = ["Invasive_Tumor", "DCIS_1", "DCIS_2", "Prolif_Invasive_Tumor"]
immune_folders = ["CD4+_T_Cells", "CD8+_T_Cells", "B_Cells", "Stromal"]
sub_label_list = tumour_folders + immune_folders
sub_label_to_index = {name: idx for idx, name in enumerate(sub_label_list)}

def get_file_label_pairs(folders, main_label):
    pairs = []
    for folder in folders:
        files = tf.io.gfile.glob(os.path.join(folder, "*.png"))
        for f in files:
            sub_label = os.path.basename(folder)
            pairs.append((f, main_label, sub_label))
    return pairs

tumour_files = get_file_label_pairs(tumour_folders, main_label=1)
immune_files = get_file_label_pairs(immune_folders, main_label=0)
all_file_label_pairs = tumour_files + immune_files
print(f"Total images before balancing: {len(all_file_label_pairs)}")

def create_balanced_oversampled_dataset(file_label_pairs):
    label_to_files = defaultdict(list)
    for f, main_label, sub_label in file_label_pairs:
        label_to_files[sub_label].append((f, main_label, sub_label))
    
    max_class_size = max(len(files) for files in label_to_files.values())
    print(f"Max class size to balance to: {max_class_size}")

    oversampled = []
    for sub_label, files in label_to_files.items():
        originals = files.copy()
        oversampled.extend((f, m, s, False) for f, m, s in originals)

        extra_needed = max_class_size - len(originals)
        augmented_files = random.choices(originals, k=extra_needed)
        oversampled.extend((f, m, s, True) for f, m, s in augmented_files)

    random.shuffle(oversampled)
    return oversampled

oversampled_pairs = create_balanced_oversampled_dataset(all_file_label_pairs)
print(f"Total images after oversampling: {len(oversampled_pairs)}")

def augment_image(img):
    angles = [0, 90, 180, 270]
    angle = random.choice(angles)
    
    if angle == 90:
        img = tf.image.rot90(img, k=1)
    elif angle == 180:
        img = tf.image.rot90(img, k=2)
    elif angle == 270:
        img = tf.image.rot90(img, k=3)

    flip_type = random.choice(['none', 'horizontal', 'vertical', 'both'])
    if flip_type == 'horizontal':
        img = tf.image.flip_left_right(img)
    elif flip_type == 'vertical':
        img = tf.image.flip_up_down(img)
    elif flip_type == 'both':
        img = tf.image.flip_left_right(img)
        img = tf.image.flip_up_down(img)

    return img

def apply_clahe(img_np):
    lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)

    limg = cv2.merge((cl, a, b))
    final = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)
    return final
    
def _load_and_preprocess(path):
    img = cv2.imread(path.numpy().decode('utf-8'))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (img_width, img_height))
    img = apply_clahe(img)
    img = img.astype(np.float32) / 255.0
    return img
    
def decode_img(img_path, augment=False):
    img = tf.py_function(_load_and_preprocess, [img_path], tf.float32)
    img.set.shape((img_width, img_height, 3))
        
    if augment:
        img = augment_image(img)
    return img

def prepare_dataset(pairs):
    file_paths, main_labels, sub_labels, augment_flags = [], [], [], []

    for f, m, s, a in pairs:
        file_paths.append(f)
        main_labels.append(m)
        sub_labels.append(sub_label_to_index[s])
        augment_flags.append(a)

    file_paths = np.array(file_paths)
    main_labels = np.array(main_labels)
    sub_labels = np.array(sub_labels)
    augment_flags = np.array(augment_flags)

    train_files, test_files, train_main, test_main, train_sub, test_sub, train_aug, test_aug = train_test_split(
        file_paths, main_labels, sub_labels, augment_flags, test_size=0.2, stratify=sub_labels, random_state=seed
    )

    def process_path(file_path, main_label, sub_label, augment_flag):
        img = tf.py_function(_load_and_preprocess, [file_path], tf.float32)
        img.set_shape((img_width, img_height, 3))
        
        img = tf.cond(
            augment_flag,
            lambda: augment_image(img),
            lambda: img
        )
    
        return img, {"binary_output": main_label, "multiclass_output": sub_label}

    train_ds = tf.data.Dataset.from_tensor_slices((train_files, train_main, train_sub, train_aug))
    train_ds = train_ds.map(process_path, num_parallel_calls=AUTOTUNE).shuffle(1000).batch(batch_size).prefetch(AUTOTUNE)

    test_ds = tf.data.Dataset.from_tensor_slices((test_files, test_main, test_sub, test_aug))
    test_ds = test_ds.map(process_path, num_parallel_calls=AUTOTUNE).batch(batch_size).prefetch(AUTOTUNE)

    return train_ds, test_ds

def build_multitask_model():
    inputs = tf.keras.Input(shape=(img_width, img_height, 3))
    x = tf.keras.layers.Conv2D(32, (3,3), activation='relu')(inputs)
    x = tf.keras.layers.MaxPooling2D((2,2))(x)
    x = tf.keras.layers.Conv2D(64, (3,3), activation='relu')(x)
    x = tf.keras.layers.MaxPooling2D((2,2))(x)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)

    binary_output = tf.keras.layers.Dense(1, activation='sigmoid', name="binary_output")(x)
    multiclass_logits = tf.keras.layers.Dense(8, name="raw_multiclass_logits")(x)
    scaled_logits = tf.keras.layers.Multiply()([multiclass_logits, binary_output])
    multiclass_output = tf.keras.layers.Activation('softmax', name="multiclass_output")(scaled_logits)

    model = tf.keras.Model(inputs=inputs, outputs=[binary_output, multiclass_output])

    model.compile(
        optimizer='adam',
        loss={
            "binary_output": "binary_crossentropy",
            "multiclass_output": "sparse_categorical_crossentropy"
        },
        metrics={
            "binary_output": "accuracy",
            "multiclass_output": "accuracy"
        }
    )

    return model

def train_and_evaluate(train_ds, test_ds, title="Model"):
    model = build_multitask_model()
    model.summary()

    history = model.fit(train_ds, epochs=10, validation_data=test_ds)

    results = model.evaluate(test_ds)
    print(f"\nTest results - {title}:")
    print(f"Binary Accuracy: {results[3]:.4f}")
    print(f"Multiclass Accuracy: {results[4]:.4f}")

    y_true, y_pred = [], []

    for images, labels in test_ds:
        preds = model.predict(images, verbose=0)
        bin_preds = preds[0].flatten()
        multi_preds = preds[1]

        true_bin = labels['binary_output'].numpy().flatten()
        true_multi = labels['multiclass_output'].numpy().flatten()

        tumour_mask = (true_bin == 1) & (bin_preds > 0.5)
    
        y_true.extend(true_multi[tumour_mask])
        y_pred.extend(np.argmax(multi_preds[tumour_mask], axis=1))

        # tumour_indices = np.where(true_bin == 1)[0]
        # if len(tumour_indices) > 0:
        #     y_true.append(true_multi[tumour_indices])
        #     y_pred.append(np.argmax(multi_preds[tumour_indices], axis=1))

    if y_true and y_pred:
        y_true_tumour = np.concatenate(y_true)
        y_pred_tumour = np.concatenate(y_pred)
    
    print("\nClassification Report (Tumour cells only):")
    print(classification_report(
        y_true_tumour, 
        y_pred_tumour, 
        labels=tumour_labels,
        target_names=tumour_folders
    ))

    cm = confusion_matrix(y_true_tumour, y_pred_tumour)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=tumour_folders, yticklabels=tumour_folders)
    plt.title(f'Confusion Matrix - {title}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

    return model

train_ds_over, test_ds_over = prepare_dataset(oversampled_pairs)

print("\n--- Training with OVERSAMPLING ---")
model_over = train_and_evaluate(train_ds_over, test_ds_over, title="Oversampling")